# hallucination-eval — Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JananiV07/hallucination-eval/blob/main/notebooks/walkthrough.ipynb)

This notebook walks through measuring LLM hallucination with three metrics:

| Metric | Question it answers | Backend |
| --- | --- | --- |
| **FactScore** | Are the answer's claims consistent with known facts? | NLI cross-encoder |
| **FaithScore** | Is every claim grounded in the supplied context? | NLI cross-encoder |
| **EntityScore** | Do the answer's named entities appear in the source? | spaCy NER |

All three return a score in `[0, 1]` where **higher = less hallucination**.

## 0. Setup

Install the package and the spaCy model. On Colab this downloads the NLI model (~280 MB) on first use.

In [ ]:
# On Colab / a fresh environment, uncomment:
# !pip install -q hallucination-eval
# !python -m spacy download en_core_web_sm -q
print('Ready. (Uncomment the installs above if running on Colab.)')

## 1. Score a single answer

We compare a **correct** answer and a **hallucinated** answer against the same context.

In [ ]:
from hallucination_eval import FactScore, FaithScore, EntityScore

context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in "
    "Paris, France. It was completed in 1889 and stands 330 metres tall."
)
question = "Where is the Eiffel Tower and when was it built?"
correct = "The Eiffel Tower is in Paris, France, and was completed in 1889."
hallucinated = "The Eiffel Tower is in Rome, Italy, and was completed in 1750."

# Build the evaluators once (models load lazily on first call).
fact, faith, entity = FactScore(), FaithScore(), EntityScore()

In [ ]:
for label, answer in [("correct", correct), ("hallucinated", hallucinated)]:
    print(f"--- {label} answer ---")
    print(f"  FactScore : {fact.evaluate(question, context, answer):.3f}")
    print(f"  FaithScore: {faith.evaluate(question, context, answer):.3f}")
    print(f"  EntityScore: {entity.evaluate(question, context, answer):.3f}")

You should see the correct answer score near **1.0** on every metric, while the hallucinated answer scores much lower (Rome/Italy/1750 are contradicted by the context and absent from its entities).

## 2. Batch scoring + a report

`evaluate_batch` returns summary stats and per-sample diagnostics. `report` turns several evaluators into one table.

In [ ]:
from hallucination_eval.report import build_report, render_report

samples = [
    {"id": "good", "question": question, "context": context, "reference": context, "answer": correct},
    {"id": "bad", "question": question, "context": context, "reference": context, "answer": hallucinated},
]

results = [fact.evaluate_batch(samples), faith.evaluate_batch(samples), entity.evaluate_batch(samples)]
report = build_report(results, meta={"model": "demo", "dataset": "manual", "n_samples": len(samples)})
render_report(report, show_samples=True)

## 3. Load a real benchmark

`load_samples` normalises HaluEval / TruthfulQA into one schema. Here we score HaluEval's built-in *right* vs *hallucinated* answers — no LLM API key required.

In [ ]:
from hallucination_eval import load_samples

samples = load_samples("halueval", limit=5)
print(f"Loaded {len(samples)} samples. First question:\n  {samples[0]['question']}\n")

# Score the gold answers (should be high) vs the hallucinated answers (should be lower).
for key, label in [("gold_answer", "GOLD"), ("hallucinated_answer", "HALLUCINATED")]:
    for s in samples:
        s["answer"] = s.get(key) or ""
    r = build_report(
        [fact.evaluate_batch(samples), faith.evaluate_batch(samples), entity.evaluate_batch(samples)],
        meta={"model": label, "dataset": "halueval", "n_samples": len(samples)},
    )
    print(f"\n===== {label} answers =====")
    render_report(r)

## 4. Plug in your own model

`ModelClient` speaks the OpenAI Chat Completions API, so any OpenAI-compatible endpoint works (OpenAI, Ollama, vLLM, …). It generates an answer per question; then you score it exactly as above.

In [ ]:
from hallucination_eval import ModelClient

# Requires OPENAI_API_KEY (gpt-4o-mini) or a running local server (gemma2 / mistral via Ollama).
# client = ModelClient("gpt-4o-mini")
# for s in samples:
#     s["answer"] = client.generate(s["question"], s.get("context"))
# render_report(build_report([fact.evaluate_batch(samples), faith.evaluate_batch(samples), entity.evaluate_batch(samples)]))
print("Set OPENAI_API_KEY (or start Ollama) and uncomment to generate-then-score.")

Or from the command line:

```bash
hallucination-eval --model gpt-4o-mini --dataset halueval --limit 50 --show-samples -o reports/run.json
```